# Merging the INE housing and tourism tables

## What this notebook does

We have 7 CSV files downloaded from the INE (Statistics Portugal). Each one has a different layout, so here we clean them one by one, put them all in the same "long" shape (one row per region and year) and join them into a single CSV: `datos_unidos.csv`.

The code is intentionally plain: one chunk per file, no fancy functions, so you can edit any step by hand without breaking the others.

### Python to R cheat sheet

| Python (pandas) | R (tidyverse) |
|----------------------------------|--------------------------------------|
| `pd.read_csv(sep=";", encoding="latin1")` | `read_delim(delim = ";", locale = locale(encoding = "latin1"))` |
| `df.melt(...)` | `pivot_longer(...)` |
| `df.pivot(...)` | `pivot_wider(...)` |
| `df.merge(other, how="left", on=[...])` | `left_join(other, by = c(...))` |
| `df.groupby("a").sum()` | `df |> group_by(a) |> summarise(...)` |
| `df[df.x > 3]` | `df |> filter(x > 3)` |
| `df["new"] = ...` | `df |> mutate(new = ...)` |
| `df.head()` / `df.info()` | `head(df)` / `glimpse(df)` |
| `df.to_csv("a.csv", index=False)` | `write_csv(df, "a.csv")` |

Note: `|>` is R's "pipe". It passes the result on the left as the first argument of the function on the right, very much like chaining `.method()` calls in pandas.

## 1. Setup

In [1]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


## 2. A small helper to read the INE files

The INE exports have a few lines of text on top, several header rows and some notes at the bottom, and those sizes change from file to file. Instead of counting lines by hand, this helper reads the whole file and keeps only the lines that look like data rows (the ones that start with a region code such as `11A: Área Metropolitana do Porto`).

In [2]:
# Reads an INE csv and returns only the data rows, as text columns (X1, X2, ...).
# We convert to numbers later, file by file.
read_ine_rows <- function(path) {
  all_lines <- read_lines(path, locale = locale(encoding = "latin1"))

  # A data row starts with a code like "PT: ", "11A: " or "1D1: ".
  # The population file has a year before the code ("2021;PT: ..."), so we allow that too.
  is_data_row <- str_detect(all_lines, "^(\\d{4})?;?[0-9A-Z]{1,4}: ")

  read_delim(
    I(all_lines[is_data_row]),
    delim = ";",
    col_names = FALSE,
    col_types = cols(.default = col_character()),
    show_col_types = FALSE
  )
}

# Turns text like "1460,8" into the number 1460.8 (the INE uses a decimal comma).
to_number <- function(x) parse_number(x, locale = locale(decimal_mark = ","))

## 3. Region lookup

Every file labels regions as `"code: name"`. We read the "new homes" file first just to build one clean list of regions. We keep the **NUTS III** level (3-character codes like `11A` or `1D1`), plus the two island regions, whose code in most files is just `20` and `30`.

In [ ]:
homes_raw <- read_ine_rows("Fogos concluídos.csv")

regions <- homes_raw |>
  select(X1) |>
  separate(X1, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  filter(nchar(nuts_code) == 3 | nuts_code %in% c("20", "30"))

# Lisbon Metro Area only exists in the tourism file (old NUTS 2013 map), so we add it by hand.
regions <- regions |>
  add_row(nuts_code = "170", region = "Área Metropolitana de Lisboa")

regions

## 4. Wide tables with one value per year

Four files have the same layout: one row per region and one column per year. For each one we name the year columns, drop the empty trailing column, melt to long format and keep only the regions we care about.

### Monthly wage

In [ ]:
wage_raw <- read_ine_rows("Ganho_médio_mensal.csv")

wage_years <- 2024:2021   # the years, in the same order as the file columns

wage <- wage_raw |>
  select(1:(length(wage_years) + 1)) |>
  set_names(c("region_raw", wage_years)) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  pivot_longer(-c(nuts_code, region), names_to = "year", values_to = "wage_monthly") |>
  mutate(year = as.integer(year), wage_monthly = to_number(wage_monthly)) |>
  select(nuts_code, year, wage_monthly)

### Median rent per m²

In [ ]:
rent_raw <- read_ine_rows("Valor_mediano_das_rendas_de_novos_contratos_de_arrendamento_de_alojamentos_familiares.csv")

rent_years <- 2025:2020

rent <- rent_raw |>
  select(1:(length(rent_years) + 1)) |>
  set_names(c("region_raw", rent_years)) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  pivot_longer(-c(nuts_code, region), names_to = "year", values_to = "rent_m2") |>
  mutate(year = as.integer(year), rent_m2 = to_number(rent_m2)) |>
  # This file lists the Azores twice (codes 20 and 200). We keep 20, like in the other files.
  filter(nuts_code != "200") |>
  select(nuts_code, year, rent_m2)

### New homes completed

In [ ]:
homes_years <- 2025:2011

homes <- homes_raw |>
  select(1:(length(homes_years) + 1)) |>
  set_names(c("region_raw", homes_years)) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  pivot_longer(-c(nuts_code, region), names_to = "year", values_to = "homes_completed") |>
  mutate(year = as.integer(year), homes_completed = to_number(homes_completed)) |>
  select(nuts_code, year, homes_completed)

### Foreign residents

In [ ]:
foreign_raw <- read_ine_rows("População_estrangeira_com_estatuto_legal_de_residente.csv")

foreign_years <- 2023:2021

foreign <- foreign_raw |>
  select(1:(length(foreign_years) + 1)) |>
  set_names(c("region_raw", foreign_years)) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  pivot_longer(-c(nuts_code, region), names_to = "year", values_to = "foreign_residents") |>
  mutate(year = as.integer(year), foreign_residents = to_number(foreign_residents)) |>
  select(nuts_code, year, foreign_residents)

## 5. Resident population (Census 2021)

This file has only one year and an extra column in front (the year, which is only filled in on the first row). So we ignore that column and set the year ourselves.

In [ ]:
population_raw <- read_ine_rows("Populacao residente.csv")

population <- population_raw |>
  select(region_raw = X2, resident_population = X3) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  mutate(
    year = 2021L,
    resident_population = to_number(resident_population)
  ) |>
  select(nuts_code, year, resident_population)

## 6. Median sale price per m² (by home size)

This one has 5 columns for every year: Total, T0 or T1, T2, T3 and T4 or more. We build the column names by hand (year + size), melt, then split them apart again so each size becomes its own column.

In [ ]:
sales_raw <- read_ine_rows("Valor_mediano_das_vendas_de_alojamentos_familiares.csv")

sales_years <- 2025:2019
sales_sizes <- c("total", "t0t1", "t2", "t3", "t4plus")

sales_col_names <- paste0(rep(sales_years, each = length(sales_sizes)), "_", sales_sizes)

sales <- sales_raw |>
  select(1:(length(sales_col_names) + 1)) |>
  set_names(c("region_raw", sales_col_names)) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  pivot_longer(-c(nuts_code, region), names_to = "year_size", values_to = "price") |>
  separate(year_size, into = c("year", "size"), sep = "_") |>
  mutate(year = as.integer(year), price = to_number(price)) |>
  pivot_wider(
    id_cols = c(nuts_code, year),
    names_from = size,
    values_from = price,
    names_prefix = "sale_price_m2_"
  )

## 7. Overnight stays (monthly -\> yearly)

The tourism file uses the older **NUTS 2013** map and has one column per month ("Agosto de 2026", "Julho de 2026", ...). Two things to do here:

1.  Add the up-to-date NUTS 2024 code for each region (matched by name).
2.  Add up the 12 months of each year. We only keep 2020-2025, because 2026 is incomplete.

In [ ]:
stays_lines <- read_lines(
  "Dormidas_nos_estabelecimentos_de_alojamento_turístico.csv",
  locale = locale(encoding = "latin1")
)

# The header line with the months is the one that starts with ";Agosto de 2026;Julho de 2026..."
month_header <- stays_lines[str_detect(stays_lines, "^;[A-Za-zç]+ de \\d{4};")]
month_names  <- str_split(month_header, ";")[[1]]
month_names  <- month_names[month_names != ""]   # drop the empty bits

stays_raw <- read_ine_rows("Dormidas_nos_estabelecimentos_de_alojamento_turístico.csv")

stays_monthly <- stays_raw |>
  select(1:(length(month_names) + 1)) |>
  set_names(c("region_raw", month_names)) |>
  separate(region_raw, into = c("nuts_code_2013", "region"), sep = ": ", extra = "merge") |>
  pivot_longer(-c(nuts_code_2013, region), names_to = "month", values_to = "tourist_nights") |>
  mutate(
    year = as.integer(str_extract(month, "\\d{4}")),
    tourist_nights = to_number(tourist_nights)
  )

Now the code translation. Left column = code in the tourism file (NUTS 2013), right column = code used by the other files (NUTS 2024). Most Northern regions and the Algarve and islands keep the same code. The Centre and Alentejo were reorganised, so those need a manual match.

In [ ]:
nuts_translation <- tribble(
  ~nuts_code_2013, ~nuts_code,
  # North (same codes)
  "111", "111", "112", "112", "119", "119", "11A", "11A",
  "11B", "11B", "11C", "11C", "11D", "11D", "11E", "11E",
  # Centre (old 16x -> new 19x, plus Oeste / Médio Tejo moved to 1D)
  "16D", "191",   # Região de Aveiro
  "16E", "192",   # Região de Coimbra
  "16F", "193",   # Região de Leiria
  "16G", "194",   # Viseu Dão Lafões
  "16H", "195",   # Beira Baixa
  "16J", "196",   # Beiras e Serra da Estrela
  "16B", "1D1",   # Oeste
  "16I", "1D2",   # Médio Tejo
  # Alentejo (old 18x -> new 1Cx, Lezíria do Tejo moved to 1D3)
  "181", "1C1",   # Alentejo Litoral
  "184", "1C2",   # Baixo Alentejo
  "186", "1C3",   # Alto Alentejo
  "187", "1C4",   # Alentejo Central
  "185", "1D3",   # Lezíria do Tejo
  # Same code in both maps
  "150", "150",   # Algarve
  "20",  "20",    # Azores
  "30",  "30",    # Madeira
  # Lisbon Metro Area: in 2024 it is split in two (1A0 + 1B0), so we keep it as its own row
  "170", "170"
)

stays <- stays_monthly |>
  inner_join(nuts_translation, by = "nuts_code_2013") |>   # also drops country / NUTS I / NUTS II rows
  filter(year >= 2020, year <= 2025) |>                   # full years only
  group_by(nuts_code, year) |>
  summarise(tourist_nights = sum(tourist_nights), .groups = "drop")

## 8. Join everything

We start from a "skeleton" with every region for every year from 2011 to 2025, and then attach each table to it. Where a table has no data for a given region and year, the value stays `NA` (the equivalent of `NaN` in pandas).

In [ ]:
skeleton <- expand_grid(regions, year = 2011:2025)

merged <- skeleton |>
  left_join(wage,       by = c("nuts_code", "year")) |>
  left_join(rent,       by = c("nuts_code", "year")) |>
  left_join(sales,      by = c("nuts_code", "year")) |>
  left_join(homes,      by = c("nuts_code", "year")) |>
  left_join(foreign,    by = c("nuts_code", "year")) |>
  left_join(population, by = c("nuts_code", "year")) |>
  left_join(stays,      by = c("nuts_code", "year")) |>
  arrange(nuts_code, year)

glimpse(merged)

## 9. Quick checks

How many rows do we have, and how many missing values per column? A lot of `NA` is expected: each source covers different years.

In [ ]:
nrow(merged)
n_distinct(merged$nuts_code)

# There must be exactly one row per region and year (this should print 0).
merged |> count(nuts_code, year) |> filter(n > 1) |> nrow()

merged |>
  summarise(across(everything(), ~ sum(is.na(.x)))) |>
  pivot_longer(everything(), names_to = "column", values_to = "missing")

Do all the regions of each table exist in our region list? Each of these should show zero rows (otherwise a code did not match).

In [ ]:
wage       |> anti_join(regions, by = "nuts_code") |> distinct(nuts_code)
rent       |> anti_join(regions, by = "nuts_code") |> distinct(nuts_code)
sales      |> anti_join(regions, by = "nuts_code") |> distinct(nuts_code)
foreign    |> anti_join(regions, by = "nuts_code") |> distinct(nuts_code)
population |> anti_join(regions, by = "nuts_code") |> distinct(nuts_code)
stays      |> anti_join(regions, by = "nuts_code") |> distinct(nuts_code)

Spot check against the original files: Porto Metropolitan Area (`11A`) in 2024 should have a monthly wage of 1620.1, a rent of 9.14 and 5804 new homes.

In [ ]:
merged |> filter(nuts_code == "11A", year == 2024)

## 10. Save the result

In [ ]:
write_csv(merged, "datos_unidos.csv")